In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Compact runtime/scalability + hidden-entry matrix accuracy table.

Creates ONE LaTeX table:

- Real 15x15 biological benchmarks:
    Hyb-Adam-UM only
    Cercopithecidae and Heterogeneous rows

- Synthetic 30x30 scalability benchmark:
    selected matrix-completion methods
    includes MW*-proj
    excludes FastME-proj
    includes separate Hyb-Adam-UM rows for 5k and 10k steps
    no tree metrics, no branch-length metrics

The table reports:
    - n
    - missingness range
    - |Omega_miss|
    - Hyb-Adam-UM optimized-variable range
    - C(n,3)
    - RMSE_miss, displayed as x 10^{-2}
    - MAE_miss, displayed as x 10^{-2}
    - Hyb-Adam-UM convergence epoch range
    - runtime range

Reviewer logic:
    The reviewer asked for more precise runtime reporting, including the
    actual number of optimized variables and average epochs to convergence.
    This compact table answers that indirectly by reporting, for Hyb-Adam-UM,
    both the optimized-variable range and convergence-epoch range over the
    same missingness levels used in the runtime ranges.

Important:
    |Omega_miss| is the common number of masked upper-triangular distance
    entries:
        |Omega_miss| = round(p * C(n,2))

    For Hyb-Adam-UM, |Omega_miss| is also the number of Adam-optimized
    variables. For MW*, NJ*, LRMC, KNN, and MDS-SMACOF, it is the common
    problem size only.

Outputs:
    /home/user/bioinformatics/!after review -runtime_table/
        runtime_scalability_compact_5k10k_long_numeric.csv
        runtime_scalability_compact_5k10k_numeric.csv
        runtime_scalability_compact_5k10k_inventory.csv
        runtime_scalability_compact_5k10k_table.tex
"""

from __future__ import annotations

import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

OUT_DIR = Path("/home/user/bioinformatics/!after review -runtime_table")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_LONG_NUMERIC = OUT_DIR / "runtime_scalability_compact_5k10k_long_numeric.csv"
OUT_COMPACT_NUMERIC = OUT_DIR / "runtime_scalability_compact_5k10k_numeric.csv"
OUT_INVENTORY = OUT_DIR / "runtime_scalability_compact_5k10k_inventory.csv"
OUT_TEX = OUT_DIR / "runtime_scalability_compact_5k10k_table.tex"

MISSING_LEVELS = [30, 50, 65, 85]

REAL_READY_ROOT = Path("/home/user/bioinformatics/!after review -ready")
REAL_HET_ROOT = Path("/home/user/bioinformatics/!after review -heterogeneous")
SYN30_ROOT = Path("/home/user/bioinformatics/!after review -synt30x30scalability")

# Matrix errors in source summaries are raw distances.
# Display them as x 10^{-2}: raw 0.0234 -> table value 2.34.
MATRIX_ERROR_DISPLAY_SCALE = 100.0

# MW summary contains stepA_method_mode = additive / ultrametric.
# This is NOT a method-name column. It is an internal MW mode.
# Use "ultrametric" if this is your MW* variant.
# Change to "additive" if you want the additive MW row instead.
MW_INTERNAL_MODE = "ultrametric"

# Set False after checking that MW appears.
DEBUG_LOAD = True

# Do not print rows with no usable data.
INCLUDE_EMPTY_METHOD_ROWS = False

NA_TEX = r"\textemdash"


# ============================================================
# Data structures
# ============================================================

@dataclass(frozen=True)
class RunSpec:
    experiment: str
    method_key: str
    method_latex: str
    n: int
    total_runs_per_missingness: int
    aliases: Sequence[str]
    files: Sequence[Path]
    is_hyb_adam: bool = False
    internal_filters: Tuple[Tuple[str, str], ...] = ()


RUNS: List[RunSpec] = [
    # --------------------------------------------------------
    # Real 15x15 Hyb-Adam-UM only
    # --------------------------------------------------------
    RunSpec(
        experiment="Cercopithecidae",
        method_key="Hyb-Adam-UM-real-close",
        method_latex=r"Hyb-Adam-UM",
        n=15,
        total_runs_per_missingness=30,
        aliases=("Hyb-Adam-UM", "HybAdamUM", "Hyb Adam UM"),
        files=(
            REAL_READY_ROOT
            / "Hyb-Adam-UM-warmbaseline-lr002-5ksteps-1start"
            / "hyb_adam_um_outputs/tables/hyb_adam_um_summary_formatted_by_missingness.csv",
            REAL_READY_ROOT
            / "Hyb-Adam-UM-warmbaseline-lr002-5ksteps-1start"
            / "hyb_adam_um_outputs/tables/hyb_adam_um_summary_numeric_by_missingness.csv",
        ),
        is_hyb_adam=True,
    ),
    RunSpec(
        experiment="Heterogeneous",
        method_key="Hyb-Adam-UM-real-heterogeneous",
        method_latex=r"Hyb-Adam-UM",
        n=15,
        total_runs_per_missingness=30,
        aliases=("Hyb-Adam-UM", "HybAdamUM", "Hyb Adam UM"),
        files=(
            REAL_HET_ROOT
            / "Hyb-Adam-UM-warmbaseline-lr002-5ksteps-1start"
            / "hyb_adam_um_outputs/tables/hyb_adam_um_summary_formatted_by_missingness.csv",
            REAL_HET_ROOT
            / "Hyb-Adam-UM-warmbaseline-lr002-5ksteps-1start"
            / "hyb_adam_um_outputs/tables/hyb_adam_um_summary_numeric_by_missingness.csv",
        ),
        is_hyb_adam=True,
    ),

    # --------------------------------------------------------
    # Synthetic 30x30 selected methods.
    # FastME-proj intentionally removed.
    # MW*-proj retained.
    # --------------------------------------------------------
    RunSpec(
        experiment="Synthetic",
        method_key="Hyb-Adam-UM-syn30-5k",
        method_latex=r"Hyb-Adam-UM (5k)",
        n=30,
        total_runs_per_missingness=5,
        aliases=("Hyb-Adam-UM", "HybAdamUM", "Hyb Adam UM"),
        files=(
            SYN30_ROOT
            / "Hyb-Adam-UM-warmbaseline-lr002-5ksteps-1start"
            / "hyb_adam_um_outputs/tables/hyb_adam_um_summary_formatted_by_missingness.csv",
            SYN30_ROOT
            / "Hyb-Adam-UM-warmbaseline-lr002-5ksteps-1start"
            / "hyb_adam_um_outputs/tables/hyb_adam_um_summary_numeric_by_missingness.csv",
        ),
        is_hyb_adam=True,
    ),
    RunSpec(
        experiment="Synthetic",
        method_key="Hyb-Adam-UM-syn30-10k",
        method_latex=r"Hyb-Adam-UM (10k)",
        n=30,
        total_runs_per_missingness=5,
        aliases=("Hyb-Adam-UM", "HybAdamUM", "Hyb Adam UM"),
        files=(
            SYN30_ROOT
            / "Hyb-Adam-UM-warmbaseline-lr002-10ksteps-1start"
            / "hyb_adam_um_outputs/tables/hyb_adam_um_summary_formatted_by_missingness.csv",
            SYN30_ROOT
            / "Hyb-Adam-UM-warmbaseline-lr002-10ksteps-1start"
            / "hyb_adam_um_outputs/tables/hyb_adam_um_summary_numeric_by_missingness.csv",
        ),
        is_hyb_adam=True,
    ),
    RunSpec(
        experiment="Synthetic",
        method_key="MW-proj-syn30",
        method_latex=r"MW$^\star$-proj",
        n=30,
        total_runs_per_missingness=5,
        aliases=(
            "MW-proj",
            "MWProj",
            "MW proj",
            "MW-Proj",
            "MW*-proj",
            "MW* proj",
            "MWstar",
            "MWstar-proj",
            "MW star proj",
            "MW-star-proj",
            r"MW$^\star$-proj",
            r"MW$^\star$ proj",
            r"MW^\star-proj",
            r"MW^\star proj",
            r"MW^{\star}-proj",
            r"MW^{\star} proj",
        ),
        files=(
            SYN30_ROOT
            / "MW-proj-afterreview/mw_proj_outputs/tables/mw_proj_summary_formatted_by_missingness.csv",
            SYN30_ROOT
            / "MW-proj-afterreview/mw_proj_outputs/tables/mw_proj_summary_numeric_by_missingness.csv",
            SYN30_ROOT
            / "MW-proj-after review/mw_proj_outputs/tables/mw_proj_summary_formatted_by_missingness.csv",
            SYN30_ROOT
            / "MW-proj-after review/mw_proj_outputs/tables/mw_proj_summary_numeric_by_missingness.csv",
        ),
        is_hyb_adam=False,
        internal_filters=(("stepA_method_mode", MW_INTERNAL_MODE),),
    ),
    RunSpec(
        experiment="Synthetic",
        method_key="NJstar-syn30",
        method_latex=r"NJ$^\star$-proj",
        n=30,
        total_runs_per_missingness=5,
        aliases=(
            "NJstar",
            "NJstar-STRICT",
            "NJ-proj",
            "NJ*",
            "NJ*-proj",
            "NJ-star",
            "NJStarStrict",
            r"NJ$^\star$-proj",
            r"NJ^\star-proj",
            r"NJ^{\star}-proj",
        ),
        files=(
            SYN30_ROOT
            / "NJ-proj-afterreview/njstar_strict_outputs/tables/njstar_strict_summary_formatted_by_missingness.csv",
            SYN30_ROOT
            / "NJ-proj-afterreview/njstar_strict_outputs/tables/njstar_strict_summary_numeric_by_missingness.csv",
            SYN30_ROOT
            / "NJ-proj-afterrewview/njstar_strict_outputs/tables/njstar_strict_summary_formatted_by_missingness.csv",
            SYN30_ROOT
            / "NJ-proj-afterrewview/njstar_strict_outputs/tables/njstar_strict_summary_numeric_by_missingness.csv",
        ),
        is_hyb_adam=False,
    ),
    RunSpec(
        experiment="Synthetic",
        method_key="LRMC-syn30",
        method_latex=r"LRMC",
        n=30,
        total_runs_per_missingness=5,
        aliases=("LRMC", "LRMC-SoftImpute", "SoftImpute", "Soft-Impute"),
        files=(
            SYN30_ROOT
            / "LRMC-after review/lrmc_softimpute_outputs/tables/lrmc_softimpute_summary_formatted_by_missingness.csv",
            SYN30_ROOT
            / "LRMC-after review/lrmc_softimpute_outputs/tables/lrmc_softimpute_summary_numeric_by_missingness.csv",
        ),
        is_hyb_adam=False,
    ),
    RunSpec(
        experiment="Synthetic",
        method_key="KNN-Impute-syn30",
        method_latex=r"KNN-impute",
        n=30,
        total_runs_per_missingness=5,
        aliases=("KNN-Impute", "KNN-impute", "KNN", "KNNImpute", "KNNimpute"),
        files=(
            SYN30_ROOT
            / "KNN-impute-after review/knn_impute_outputs/tables/knn_impute_summary_formatted_by_missingness.csv",
            SYN30_ROOT
            / "KNN-impute-after review/knn_impute_outputs/tables/knn_impute_summary_numeric_by_missingness.csv",
        ),
        is_hyb_adam=False,
    ),
    RunSpec(
        experiment="Synthetic",
        method_key="MDS-SMACOF-syn30",
        method_latex=r"MDS-SMACOF",
        n=30,
        total_runs_per_missingness=5,
        aliases=("MDS-SMACOF", "MDS SMACOF", "MDSSMACOF", "MDS", "SMACOF"),
        files=(
            SYN30_ROOT
            / "MDS-SMACOF-after review/mds_smacof_outputs/tables/mds_smacof_summary_formatted_by_missingness.csv",
            SYN30_ROOT
            / "MDS-SMACOF-after review/mds_smacof_outputs/tables/mds_smacof_summary_numeric_by_missingness.csv",
        ),
        is_hyb_adam=False,
    ),
]


# ============================================================
# Text and numeric parsing
# ============================================================

def norm_text(s: str) -> str:
    s = str(s).lower()

    replacements = {
        r"\star": "star",
        r"^\star": "star",
        r"^{\star}": "star",
        "$": "",
        "{": "",
        "}": "",
        "\\": "",
        "^": "",
        "_": "",
        "*": "star",
    }

    for old, new in replacements.items():
        s = s.replace(old, new)

    return re.sub(r"[^a-z0-9]+", "", s)


def latex_escape_text(s: str) -> str:
    return (
        str(s)
        .replace("&", r"\&")
        .replace("%", r"\%")
        .replace("_", r"\_")
        .replace("#", r"\#")
    )


def choose_existing_path(paths: Sequence[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None


def clean_cell(x) -> str:
    """
    Clean a cell before numeric parsing.

    This intentionally does NOT evaluate strings such as 10^{-2}. Earlier
    code tried to parse powers and could misinterpret ordinary decimals,
    producing OverflowError. Scaling is handled explicitly elsewhere.
    """
    if pd.isna(x):
        return ""

    s = str(x).strip()

    replacements = {
        r"\pm": "±",
        "+/-": "±",
        "+-": "±",
        "−": "-",
        "–": "-",
        "—": "-",
        "$": "",
        "{": "",
        "}": "",
        ",": "",
        r"\times": " times ",
        r"\cdot": " times ",
        r"\best": "",
        r"\textbf": "",
    }

    for old, new in replacements.items():
        s = s.replace(old, new)

    return s.strip()


def first_float(x) -> float:
    """
    Return the first ordinary floating-point number in a string.
    Does not parse powers such as 10^{-2}.
    """
    s = clean_cell(x)

    if not s:
        return np.nan

    if s.upper() in {"N/A", "NA"} or s.lower() in {"nan", "none", "-"}:
        return np.nan

    m = re.search(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?", s)
    if not m:
        return np.nan

    try:
        return float(m.group(0))
    except Exception:
        return np.nan


def parse_mean_std(x) -> Tuple[float, float, str]:
    raw = "" if pd.isna(x) else str(x).strip()
    s = clean_cell(x)

    if not s:
        return np.nan, np.nan, raw

    if "±" in s:
        left, right = s.split("±", 1)
        return first_float(left), first_float(right), raw

    return first_float(s), np.nan, raw


def find_col_exact_norm(row_or_df, candidates: Sequence[str]) -> Optional[str]:
    if isinstance(row_or_df, pd.DataFrame):
        columns = list(row_or_df.columns)
    else:
        columns = list(row_or_df.index)

    col_map = {norm_text(c): c for c in columns}

    for cand in candidates:
        key = norm_text(cand)
        if key in col_map:
            return col_map[key]

    return None


def find_col_fuzzy(row_or_df, candidates: Sequence[str]) -> Optional[str]:
    """
    Fuzzy column detection for metrics, not for method columns.
    """
    exact = find_col_exact_norm(row_or_df, candidates)
    if exact is not None:
        return exact

    if isinstance(row_or_df, pd.DataFrame):
        columns = list(row_or_df.columns)
    else:
        columns = list(row_or_df.index)

    norm_cols = [(norm_text(c), c) for c in columns]
    norm_cands = [norm_text(c) for c in candidates]

    for cand_norm in norm_cands:
        if not cand_norm:
            continue
        for col_norm, original_col in norm_cols:
            if cand_norm in col_norm or col_norm in cand_norm:
                return original_col

    return None


def get_genuine_method_column(df: pd.DataFrame) -> Optional[str]:
    """
    Return a true method-name column only.

    Do NOT treat columns such as stepA_method_mode as method columns.
    Those are internal MW settings and must not be used for method filtering.
    """
    for c in df.columns:
        if norm_text(c) == "method":
            return c

    return None


def detect_pct_column(df: pd.DataFrame) -> str:
    candidates = [
        "% Missing",
        "pct_missing",
        "missingness",
        "Missingness",
        "missingness_level",
        "missingness_actual",
        "p",
    ]

    col = find_col_exact_norm(df, candidates)
    if col is not None:
        return col

    for c in df.columns:
        if "missing" in norm_text(c):
            return c

    raise ValueError(f"Cannot find missingness column. Columns: {list(df.columns)}")


def normalize_pct_value(x) -> int:
    if pd.isna(x):
        return -1

    s = str(x).strip().replace("%", "")
    v = first_float(s)

    if not np.isfinite(v):
        return -1

    if 0 < v < 1:
        v *= 100

    return int(round(v))


def get_metric(row: Optional[pd.Series], candidates: Sequence[str]) -> Tuple[float, float, str, str]:
    """
    Extract metric from:
    1) direct formatted column, e.g. runtime_seconds = '21.47 ± 0.18';
    2) numeric mean/std columns, e.g. runtime_seconds_mean/runtime_seconds_std.
    """
    if row is None:
        return np.nan, np.nan, "", ""

    direct_col = find_col_fuzzy(row, candidates)

    if direct_col is not None:
        mean, std, raw = parse_mean_std(row[direct_col])
        if np.isfinite(mean) or raw not in {"", "nan", "N/A"}:
            return mean, std, raw, direct_col

    for cand in candidates:
        mean_col = find_col_fuzzy(row, [cand + "_mean", cand + " mean", cand + ".mean"])
        std_col = find_col_fuzzy(row, [cand + "_std", cand + " std", cand + ".std"])

        if mean_col is not None:
            mean = first_float(row[mean_col])
            std = first_float(row[std_col]) if std_col is not None else np.nan
            raw = str(row[mean_col])
            return mean, std, raw, mean_col

    return np.nan, np.nan, "", ""


def set_metric(
    rec: Dict[str, object],
    prefix: str,
    row: Optional[pd.Series],
    candidates: Sequence[str],
) -> None:
    mean, std, raw, col = get_metric(row, candidates)
    rec[prefix + "_mean"] = mean
    rec[prefix + "_std"] = std
    rec[prefix + "_raw"] = raw
    rec[prefix + "_source_col"] = col


# ============================================================
# Filtering
# ============================================================

def filter_method_rows(df: pd.DataFrame, aliases: Sequence[str]) -> pd.DataFrame:
    """
    Filter by method only when the file has a genuine method column.

    If no genuine method column exists, assume the whole file corresponds to
    the RunSpec method. This is necessary for MW-proj, where the file contains
    stepA_method_mode = additive/ultrametric, but no actual method-name column.
    """
    method_col = get_genuine_method_column(df)

    if method_col is None:
        return df.copy()

    aliases_norm = {norm_text(a) for a in aliases}
    method_norm = df[method_col].map(norm_text)

    mask_exact = method_norm.isin(aliases_norm)
    if mask_exact.any():
        return df[mask_exact].copy()

    mask_fuzzy = method_norm.map(
        lambda m: any(a in m or m in a for a in aliases_norm if a and m)
    )
    if mask_fuzzy.any():
        return df[mask_fuzzy].copy()

    if df[method_col].nunique(dropna=True) == 1:
        return df.copy()

    return df.iloc[0:0].copy()


def apply_internal_filters(
    df: pd.DataFrame,
    internal_filters: Tuple[Tuple[str, str], ...],
) -> pd.DataFrame:
    """
    Apply method-specific internal filters.

    Example:
        MW-proj has stepA_method_mode = additive / ultrametric.
        The final row should not be chosen arbitrarily, so this function
        selects the configured MW_INTERNAL_MODE.
    """
    out = df.copy()

    for col_name, desired_value in internal_filters:
        col = find_col_exact_norm(out, [col_name])

        if col is None:
            if DEBUG_LOAD:
                print(f"Internal filter column not found: {col_name}")
            continue

        desired_norm = norm_text(desired_value)
        values_norm = out[col].astype(str).map(norm_text)

        mask = values_norm.eq(desired_norm)

        if DEBUG_LOAD:
            print(f"INTERNAL FILTER: {col} == {desired_value}")
            print("AVAILABLE VALUES:", sorted(out[col].dropna().astype(str).unique()))
            print("ROWS BEFORE INTERNAL FILTER:", len(out))
            print("ROWS MATCHING INTERNAL FILTER:", int(mask.sum()))

        if mask.any():
            out = out[mask].copy()
        else:
            print(
                f"WARNING: no rows match internal filter {col_name}={desired_value}. "
                f"Keeping unfiltered rows."
            )

    return out


def debug_summary_file(
    path: Optional[Path],
    df: Optional[pd.DataFrame],
    aliases: Sequence[str],
    stage: str,
) -> None:
    if not DEBUG_LOAD:
        return

    print("\n" + "=" * 80)
    print(stage)
    print("PATH:", path)

    if path is None:
        print("No existing path selected.")
        return

    if df is None:
        print("No dataframe loaded.")
        return

    print("COLUMNS:", list(df.columns))

    method_col = get_genuine_method_column(df)
    if method_col is not None:
        print("GENUINE METHOD COL:", method_col)
        print("METHOD VALUES:", sorted(df[method_col].dropna().astype(str).unique()))
        print("METHOD VALUES NORM:", sorted(df[method_col].dropna().astype(str).map(norm_text).unique()))
        print("ALIASES NORM:", sorted(norm_text(a) for a in aliases))
    else:
        print("No genuine method column found. Treating entire file as this RunSpec method.")

    try:
        pct_col = detect_pct_column(df)
        print("PCT COL:", pct_col)
        print("PCT RAW VALUES:", sorted(df[pct_col].dropna().astype(str).unique()))
        print("PCT NORMALIZED:", sorted(df[pct_col].map(normalize_pct_value).dropna().unique()))
    except Exception as exc:
        print("Could not detect pct column:", repr(exc))


def load_summary_by_pct(
    path: Optional[Path],
    aliases: Sequence[str],
    internal_filters: Tuple[Tuple[str, str], ...],
) -> Dict[int, pd.Series]:
    if path is None or not path.exists():
        if DEBUG_LOAD:
            print("\nPATH NOT FOUND:", path)
        return {}

    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]

    debug_summary_file(path, df, aliases, "LOADED RAW CSV")

    df = filter_method_rows(df, aliases)

    if DEBUG_LOAD:
        print("ROWS AFTER METHOD FILTER:", len(df))

    if df.empty:
        return {}

    df = apply_internal_filters(df, internal_filters)

    if DEBUG_LOAD:
        print("ROWS AFTER INTERNAL FILTER:", len(df))

    if df.empty:
        return {}

    pct_col = detect_pct_column(df)
    df["_pct"] = df[pct_col].map(normalize_pct_value)

    if DEBUG_LOAD:
        print("ROWS BEFORE PCT FILTER:", len(df))
        print("PCT VALUES AFTER NORMALIZATION:", sorted(df["_pct"].dropna().unique()))

    df = df[df["_pct"].isin(MISSING_LEVELS)].copy()

    if DEBUG_LOAD:
        print("ROWS AFTER PCT FILTER:", len(df))

    out: Dict[int, pd.Series] = {}
    for pct, sub in df.groupby("_pct", sort=True):
        out[int(pct)] = sub.iloc[0]

    return out


# ============================================================
# Problem-size helpers
# ============================================================

def omega_miss_count(n: int, pct: int) -> int:
    return int(round((pct / 100.0) * math.comb(n, 2)))


# ============================================================
# Build long table
# ============================================================

def build_long_table() -> Tuple[pd.DataFrame, pd.DataFrame]:
    records: List[Dict[str, object]] = []
    inventory: List[Dict[str, object]] = []

    for spec in RUNS:
        path = choose_existing_path(spec.files)

        inventory.append(
            {
                "experiment": spec.experiment,
                "method_key": spec.method_key,
                "found": path is not None,
                "used_path": str(path) if path is not None else "",
                "candidate_paths": " | ".join(str(p) for p in spec.files),
                "internal_filters": " | ".join(
                    f"{col}={val}" for col, val in spec.internal_filters
                ),
            }
        )

        summary_by_pct = load_summary_by_pct(
            path=path,
            aliases=spec.aliases,
            internal_filters=spec.internal_filters,
        )

        for pct in MISSING_LEVELS:
            row = summary_by_pct.get(pct)

            n_pairs = math.comb(spec.n, 2)
            n_triplets = math.comb(spec.n, 3)
            omega = omega_miss_count(spec.n, pct)

            rec: Dict[str, object] = {
                "experiment": spec.experiment,
                "method_key": spec.method_key,
                "method_latex": spec.method_latex,
                "n": spec.n,
                "n_pairs": n_pairs,
                "n_triplets": n_triplets,
                "pct_missing": pct,
                "omega_miss": omega,
                "summary_path": str(path) if path is not None else "",
                "row_available": row is not None,
                "default_total_runs": spec.total_runs_per_missingness,
                "is_hyb_adam": spec.is_hyb_adam,
                "internal_filters": " | ".join(
                    f"{col}={val}" for col, val in spec.internal_filters
                ),
            }

            # -------------------------
            # Hidden-entry accuracy
            # -------------------------
            set_metric(
                rec,
                "rmse_miss",
                row,
                [
                    "rmse_miss",
                    "RMSE_miss",
                    "RMSE miss",
                    "RMSE_missing",
                    "RMSE hidden",
                    "hidden_rmse",
                    "rmse_hidden",
                    r"RMSE$_{\rm miss}$",
                    r"RMSE$_{\rm miss}$ ($\times 10^{-2}$)",
                    r"RMSE$_{\mathrm{miss}}$",
                ],
            )

            set_metric(
                rec,
                "mae_miss",
                row,
                [
                    "mae_miss",
                    "MAE_miss",
                    "MAE miss",
                    "MAE_missing",
                    "MAE hidden",
                    "hidden_mae",
                    "mae_hidden",
                    r"MAE$_{\rm miss}$",
                    r"MAE$_{\rm miss}$ ($\times 10^{-2}$)",
                    r"MAE$_{\mathrm{miss}}$",
                ],
            )

            # -------------------------
            # Runtime
            # -------------------------
            set_metric(
                rec,
                "runtime_seconds",
                row,
                [
                    "runtime_seconds",
                    "Runtime seconds",
                    "Time(s)",
                    "Time",
                    "runtime",
                    "Runtime",
                ],
            )

            # -------------------------
            # Hyb-Adam-UM reviewer diagnostics
            # -------------------------
            set_metric(
                rec,
                "optimized_variables",
                row,
                [
                    "optimized_variables",
                    "Optimized variables",
                    "n_optimized",
                    "n optimized",
                ],
            )

            set_metric(
                rec,
                "n_missing_reported",
                row,
                ["n_missing", "N_missing", "missing_entries"],
            )

            set_metric(
                rec,
                "convergence_epoch",
                row,
                ["convergence_epoch", "Convergence epoch"],
            )

            set_metric(
                rec,
                "best_epoch",
                row,
                ["best_epoch", "Best epoch"],
            )

            set_metric(
                rec,
                "epochs_used",
                row,
                ["epochs_used", "Epochs used", "epochs", "Epochs"],
            )

            # -------------------------
            # Replicate/success diagnostics kept in numeric CSV
            # -------------------------
            set_metric(rec, "n_runs", row, ["n_runs", "N_runs", "n_replicates", "n_total"])
            set_metric(rec, "n_success", row, ["n_success", "N_success"])
            set_metric(rec, "n_failed", row, ["n_failed", "N_failed"])

            records.append(rec)

    return pd.DataFrame(records), pd.DataFrame(inventory)


# ============================================================
# Formatting helpers
# ============================================================

def finite_values(series: pd.Series) -> np.ndarray:
    vals = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    return vals


def format_number(
    x: float,
    decimals: int = 2,
    integer: bool = False,
    small_threshold: Optional[float] = None,
) -> str:
    if not np.isfinite(x):
        return NA_TEX

    x = float(x)

    if small_threshold is not None and 0 <= abs(x) < small_threshold:
        return rf"$<{small_threshold:.2f}$"

    if integer:
        return str(int(round(x)))

    return f"{x:.{decimals}f}"


def format_range(
    values: Sequence[float],
    decimals: int = 2,
    integer: bool = False,
    small_threshold: Optional[float] = None,
) -> str:
    arr = np.asarray(list(values), dtype=float)
    arr = arr[np.isfinite(arr)]

    if len(arr) == 0:
        return NA_TEX

    lo = float(np.min(arr))
    hi = float(np.max(arr))

    if small_threshold is not None and 0 <= abs(hi) < small_threshold:
        return rf"$<{small_threshold:.2f}$"

    if abs(lo - hi) <= max(1e-12, abs(lo) * 1e-10):
        return format_number(
            lo,
            decimals=decimals,
            integer=integer,
            small_threshold=small_threshold,
        )

    return (
        format_number(
            lo,
            decimals=decimals,
            integer=integer,
            small_threshold=small_threshold,
        )
        + "--"
        + format_number(
            hi,
            decimals=decimals,
            integer=integer,
            small_threshold=small_threshold,
        )
    )


def choose_hyb_epoch_value(row: pd.Series) -> float:
    """
    For the compact reviewer-relevant column, prefer convergence_epoch.
    Fall back to best_epoch, then epochs_used.
    """
    for metric in ["convergence_epoch", "best_epoch", "epochs_used"]:
        val = row.get(metric + "_mean", np.nan)
        if np.isfinite(val):
            return float(val)

    return np.nan


def choose_optimized_variables(row: pd.Series) -> float:
    """
    Prefer reported optimized_variables. Fall back to n_missing, then omega_miss.
    """
    val = row.get("optimized_variables_mean", np.nan)
    if np.isfinite(val):
        return float(val)

    val = row.get("n_missing_reported_mean", np.nan)
    if np.isfinite(val):
        return float(val)

    val = row.get("omega_miss", np.nan)
    if np.isfinite(val):
        return float(val)

    return np.nan


# ============================================================
# Compact table construction
# ============================================================

def compact_one_group(group: pd.DataFrame) -> Optional[Dict[str, object]]:
    group = group.sort_values("pct_missing").copy()

    experiment = str(group["experiment"].iloc[0])
    method_key = str(group["method_key"].iloc[0])
    method_latex = str(group["method_latex"].iloc[0])
    n = int(group["n"].iloc[0])
    n_triplets = int(group["n_triplets"].iloc[0])
    is_hyb_adam = bool(group["is_hyb_adam"].iloc[0])
    internal_filters = str(group["internal_filters"].iloc[0])

    available = group[group["row_available"] == True].copy()

    if available.empty and not INCLUDE_EMPTY_METHOD_ROWS:
        return None

    if available.empty:
        pcts: List[int] = []
    else:
        pcts = [int(x) for x in available["pct_missing"].tolist()]

    if len(pcts) == 0:
        missingness_txt = NA_TEX
    elif len(pcts) == 1:
        missingness_txt = str(pcts[0]) + r"\%"
    else:
        missingness_txt = str(min(pcts)) + "--" + str(max(pcts)) + r"\%"

    omega_vals = finite_values(available["omega_miss"]) if not available.empty else np.array([])

    opt_var_vals: List[float] = []
    if is_hyb_adam and not available.empty:
        for _, row in available.iterrows():
            opt_var_vals.append(choose_optimized_variables(row))

    rmse_vals_raw = finite_values(available["rmse_miss_mean"]) if not available.empty else np.array([])
    mae_vals_raw = finite_values(available["mae_miss_mean"]) if not available.empty else np.array([])

    rmse_vals_display = rmse_vals_raw * MATRIX_ERROR_DISPLAY_SCALE
    mae_vals_display = mae_vals_raw * MATRIX_ERROR_DISPLAY_SCALE

    time_vals = finite_values(available["runtime_seconds_mean"]) if not available.empty else np.array([])

    hyb_epoch_vals: List[float] = []
    if is_hyb_adam and not available.empty:
        for _, row in available.iterrows():
            hyb_epoch_vals.append(choose_hyb_epoch_value(row))

    return {
        "experiment": experiment,
        "method_key": method_key,
        "method_latex": method_latex,
        "n": n,
        "missingness": missingness_txt,
        "omega_miss_range": format_range(omega_vals, integer=True),
        "optimized_variables_range": (
            format_range(opt_var_vals, integer=True) if is_hyb_adam else NA_TEX
        ),
        "n_triplets": n_triplets,
        "rmse_miss_range": format_range(rmse_vals_display, decimals=2, integer=False),
        "mae_miss_range": format_range(mae_vals_display, decimals=2, integer=False),
        "hyb_epoch_range": (
            format_range(hyb_epoch_vals, decimals=0, integer=True) if is_hyb_adam else NA_TEX
        ),
        "time_range": format_range(
            time_vals,
            decimals=2,
            integer=False,
            small_threshold=0.01,
        ),
        "available_levels": ",".join(str(p) for p in pcts),
        "internal_filters": internal_filters,
    }


def build_compact_table(long_df: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []

    for _, group in long_df.groupby(["experiment", "method_key"], sort=False):
        rec = compact_one_group(group)
        if rec is not None:
            rows.append(rec)

    compact_df = pd.DataFrame(rows)

    if compact_df.empty:
        return compact_df

    experiment_order = {
        "Cercopithecidae": 0,
        "Heterogeneous": 1,
        "Synthetic": 2,
    }

    method_order = {
        "Hyb-Adam-UM-real-close": 0,
        "Hyb-Adam-UM-real-heterogeneous": 0,
        "Hyb-Adam-UM-syn30-5k": 0,
        "Hyb-Adam-UM-syn30-10k": 1,
        "MW-proj-syn30": 2,
        "NJstar-syn30": 3,
        "LRMC-syn30": 4,
        "KNN-Impute-syn30": 5,
        "MDS-SMACOF-syn30": 6,
    }

    compact_df["_exp_order"] = compact_df["experiment"].map(experiment_order).fillna(99)
    compact_df["_meth_order"] = compact_df["method_key"].map(method_order).fillna(99)

    compact_df = compact_df.sort_values(["_exp_order", "_meth_order"]).drop(
        columns=["_exp_order", "_meth_order"]
    )

    return compact_df


# ============================================================
# LaTeX table
# ============================================================

def make_latex_table(compact_df: pd.DataFrame) -> str:
    lines: List[str] = []

    lines.append(r"% Requires: \usepackage{booktabs,graphicx}")
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(
        r"\caption{Runtime, scalability, and hidden-entry matrix accuracy for "
        r"matrix completion. For the biological $15\times15$ benchmarks, "
        r"diagnostics are reported for Hyb-Adam-UM only because the full "
        r"matrix- and tree-level comparison is given in "
        r"Table~\ref{tab:combined_main_results}. The synthetic $30\times30$ "
        r"experiment compares the available matrix-completion methods, including "
        r"separate Hyb-Adam-UM runs with 5k and 10k optimization steps, using "
        r"five masks per missingness level and is used as a computational "
        r"scalability and reconstruction-accuracy benchmark.}"
    )
    lines.append(r"\label{tab:runtime_scalability}")
    lines.append(r"\scriptsize")
    lines.append(r"\setlength{\tabcolsep}{1.8pt}")
    lines.append(r"\resizebox{\textwidth}{!}{%")
    lines.append(r"\begin{tabular}{llcccccccccc}")
    lines.append(r"\toprule")
    lines.append(
        r"Experiment & Method & $n$ & Missingness & $|\Omega_{\rm miss}|$ & "
        r"Hyb opt. vars & $\binom{n}{3}$ & "
        r"RMSE$_{\rm miss}$ ($\times 10^{-2}$) & "
        r"MAE$_{\rm miss}$ ($\times 10^{-2}$) & "
        r"Hyb conv. epoch & Time(s) \\"
    )
    lines.append(r"\midrule")

    last_experiment = None

    for _, row in compact_df.iterrows():
        experiment = str(row["experiment"])

        if last_experiment is not None and experiment != last_experiment:
            lines.append(r"\midrule")

        last_experiment = experiment

        line = " & ".join(
            [
                latex_escape_text(row["experiment"]),
                str(row["method_latex"]),
                str(row["n"]),
                str(row["missingness"]),
                str(row["omega_miss_range"]),
                str(row["optimized_variables_range"]),
                str(row["n_triplets"]),
                str(row["rmse_miss_range"]),
                str(row["mae_miss_range"]),
                str(row["hyb_epoch_range"]),
                str(row["time_range"]),
            ]
        ) + r" \\"

        lines.append(line)

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}%")
    lines.append(r"}")
    lines.append(r"\vspace{2pt}")
    lines.append(
        r"\parbox{0.95\textwidth}{\scriptsize "
        r"The column $|\Omega_{\rm miss}|$ gives the number of masked "
        r"upper-triangular distance entries and therefore the common missing-entry "
        r"problem size. For Hyb-Adam-UM, this number is also the number of "
        r"Adam-optimized variables, reported explicitly in the ``Hyb opt. vars'' "
        r"column; for the other methods the corresponding entry is not applicable. "
        r"RMSE$_{\rm miss}$ and MAE$_{\rm miss}$ are computed only on artificially "
        r"hidden entries and are reported as $\times 10^{-2}$. The synthetic "
        r"benchmark includes separate Hyb-Adam-UM rows for 5k and 10k "
        r"optimization steps. For $n=15$, "
        r"$\binom{n}{2}=105$ and $\binom{n}{3}=455$; for $n=30$, "
        r"$\binom{n}{2}=435$ and $\binom{n}{3}=4060$. Ranges summarize the "
        r"values over the available missingness levels 30\%, 50\%, 65\%, and "
        r"85\%. The Hyb-Adam-UM convergence epoch is reported as a range over "
        r"the same missingness levels, connecting the empirical timing values "
        r"with both the number of optimized variables and the optimization "
        r"horizon.}"
    )
    lines.append(r"\end{table}")

    return "\n".join(lines)


# ============================================================
# Main
# ============================================================

def main() -> None:
    long_df, inventory_df = build_long_table()
    compact_df = build_compact_table(long_df)

    long_df.to_csv(OUT_LONG_NUMERIC, index=False)
    compact_df.to_csv(OUT_COMPACT_NUMERIC, index=False)
    inventory_df.to_csv(OUT_INVENTORY, index=False)

    latex = make_latex_table(compact_df)
    OUT_TEX.write_text(latex, encoding="utf-8")

    print("\n" + "=" * 80)
    print(f"Saved long numeric table:    {OUT_LONG_NUMERIC}")
    print(f"Saved compact numeric table: {OUT_COMPACT_NUMERIC}")
    print(f"Saved inventory:             {OUT_INVENTORY}")
    print(f"Saved LaTeX table:           {OUT_TEX}")

    missing_files = inventory_df[inventory_df["found"] == False]
    if not missing_files.empty:
        print("\nMissing summary files:")
        for _, row in missing_files.iterrows():
            print(f"  {row['experiment']} / {row['method_key']}")
            print(f"    candidates: {row['candidate_paths']}")

    if not compact_df.empty:
        shown_keys = set(compact_df["method_key"].astype(str))
    else:
        shown_keys = set()

    expected_keys = {spec.method_key for spec in RUNS}
    skipped_keys = sorted(expected_keys - shown_keys)

    if skipped_keys:
        print("\nSkipped methods with no usable rows in the LaTeX table:")
        for key in skipped_keys:
            print(f"  {key}")

    print("\nCompact rows:")
    if compact_df.empty:
        print("No compact rows were produced.")
    else:
        print(compact_df.to_string(index=False))

    print("\nLaTeX table:\n")
    print(latex)


if __name__ == "__main__":
    main()


LOADED RAW CSV
PATH: /home/user/bioinformatics/!after review -ready/Hyb-Adam-UM-warmbaseline-lr002-5ksteps-1start/hyb_adam_um_outputs/tables/hyb_adam_um_summary_formatted_by_missingness.csv
COLUMNS: ['% Missing', 'n_runs', 'n_success', 'n_failed', 'n_objective_improved', 'n_not_improved', 'missingness_actual', 'n_missing', 'n_observed', 'optimized_variables', 'RMSE_miss', 'MAE_miss', 'Pearson_miss', 'Spearman_miss', 'runtime_seconds', 'Delta_init', 'Delta_final', 'Delta_reduction_percent', 'best_epoch', 'epochs_used', 'convergence_epoch', 'final_learning_rate', 'n_restarts', 'best_restart', 'best_restart_initial_Delta', 'best_restart_final_Delta', 'upper_bound', 'initial_global_mean', 'initial_global_median', 'observed_max_distance', 'Delta_total_completed', 'Delta_normalized_completed', 'Delta_per_triangle_completed', 'Delta_relative_to_original', 'max_abs_error_observed', 'mean_abs_error_observed']
No genuine method column found. Treating entire file as this RunSpec method.
PCT COL: